# Feature Engineering

This notebook applies the same feature function to each development season for both logistic regression and CatBoost.

The 2024-25 final holdout is not loaded or transformed during this phase.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.features import (
    CONTEXT_CATEGORICAL_FEATURES,
    CONTEXT_NUMERIC_FEATURES,
    IDENTITY_CATEGORICAL_FEATURES,
    LEAKAGE_COLUMNS,
    build_features,
    extract_target,
    feature_groups,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

## 1. Feature set

The main feature set uses only shot context available when the attempt occurs. Player and team IDs are excluded by default because new players appear over time and identity can encourage memorization. They are added only for a separate model comparison.

In [2]:
feature_dictionary = pd.DataFrame(
    [
        ("shot_distance_feet", "numeric", "Reported distance from the basket"),
        ("shot_distance_squared", "numeric", "Simple nonlinear distance term for logistic regression"),
        ("loc_x_feet", "numeric", "Horizontal court location converted to feet"),
        ("loc_y_feet", "numeric", "Vertical court location converted to feet"),
        ("abs_loc_x_feet", "numeric", "Distance from the court centerline"),
        ("shot_angle_degrees", "numeric", "Absolute angle away from the basket centerline"),
        ("period", "numeric", "Game period"),
        ("seconds_remaining_in_period", "numeric", "Game-clock seconds left in the current period"),
        ("final_five_seconds", "numeric", "Indicator for attempts with five or fewer period seconds"),
        ("is_overtime", "numeric", "Indicator for periods after the fourth"),
        ("is_corner_three", "numeric", "Indicator for left- or right-corner threes"),
        ("action_type", "categorical", "Shot action label"),
        ("shot_type", "categorical", "Two- or three-point attempt"),
        ("shot_zone_basic", "categorical", "Broad court zone"),
        ("shot_zone_area", "categorical", "Left, center, or right court area"),
        ("shot_zone_range", "categorical", "Published distance band"),
    ],
    columns=["feature", "type", "reason"],
)
feature_dictionary

,feature,type,reason
0,shot_distance_feet,numeric,Reported distance from the basket
1,shot_distance_squared,numeric,Simple nonlinear distance term for logistic re...
2,loc_x_feet,numeric,Horizontal court location converted to feet
3,loc_y_feet,numeric,Vertical court location converted to feet
4,abs_loc_x_feet,numeric,Distance from the court centerline
5,shot_angle_degrees,numeric,Absolute angle away from the basket centerline
6,period,numeric,Game period
7,seconds_remaining_in_period,numeric,Game-clock seconds left in the current period
8,final_five_seconds,numeric,Indicator for attempts with five or fewer peri...
9,is_overtime,numeric,Indicator for periods after the fourth


## 2. Apply the same function to each development season

In [3]:
season_labels = {2021: "2021-22", 2022: "2022-23", 2023: "2023-24"}
raw_by_season = {}
features_by_season = {}
target_by_season = {}

for year, label in season_labels.items():
    raw_shots = pd.read_parquet(PROJECT_ROOT / "data" / "raw" / f"{year}.parquet")
    raw_by_season[label] = raw_shots
    features_by_season[label] = build_features(raw_shots)
    target_by_season[label] = extract_target(raw_shots)

features_by_season["2021-22"].head()

,shot_distance_feet,shot_distance_squared,loc_x_feet,loc_y_feet,abs_loc_x_feet,shot_angle_degrees,period,seconds_remaining_in_period,final_five_seconds,is_overtime,is_corner_three,action_type,shot_type,shot_zone_basic,shot_zone_area,shot_zone_range
0,12.000,144.000,0.400,12.000,0.400,1.909,1,677,0,0,0,Pullup Jump shot,2PT Field Goal,In The Paint (Non-RA),Center(C),8-16 ft.
1,21.000,441.000,5.000,21.200,5.000,13.271,1,648,0,0,0,Running Pull-Up Jump Shot,2PT Field Goal,Mid-Range,Center(C),16-24 ft.
2,3.000,9.000,3.700,1.500,3.700,67.932,1,566,0,0,0,Driving Layup Shot,2PT Field Goal,Restricted Area,Center(C),Less Than 8 ft.
3,16.000,256.000,-14.500,8.400,14.500,59.916,1,526,0,0,0,Jump Shot,2PT Field Goal,Mid-Range,Left Side(L),16-24 ft.
4,23.000,529.000,22.800,6.400,22.800,74.320,1,494,0,0,1,Jump Shot,3PT Field Goal,Right Corner 3,Right Side(R),24+ ft.


## 3. Verify schema, row order, and leakage protections

A shared column list is not enough: column order, data types, row indices, missingness, and target separation must also remain consistent.

In [4]:
reference_features = features_by_season["2021-22"]
reference_schema = list(zip(reference_features.columns, reference_features.dtypes.astype(str)))
schema_records = []

for label in season_labels.values():
    raw_shots = raw_by_season[label]
    feature_frame = features_by_season[label]
    target = target_by_season[label]
    current_schema = list(zip(feature_frame.columns, feature_frame.dtypes.astype(str)))

    schema_records.append(
        {
            "season": label,
            "rows": len(feature_frame),
            "features": feature_frame.shape[1],
            "schema_matches": current_schema == reference_schema,
            "index_matches_source": feature_frame.index.equals(raw_shots.index),
            "target_index_matches": target.index.equals(raw_shots.index),
            "missing_feature_values": int(feature_frame.isna().sum().sum()),
            "leakage_columns_present": bool(set(feature_frame.columns).intersection(LEAKAGE_COLUMNS)),
        }
    )

schema_audit = pd.DataFrame(schema_records).set_index("season")

assert schema_audit["schema_matches"].all()
assert schema_audit["index_matches_source"].all()
assert schema_audit["target_index_matches"].all()
assert schema_audit["missing_feature_values"].eq(0).all()
assert ~schema_audit["leakage_columns_present"].any()
schema_audit

,rows,features,schema_matches,index_matches_source,target_index_matches,missing_feature_values,leakage_columns_present
season,,,,,,,
2021-22,216722,16,True,True,True,0,False
2022-23,217218,16,True,True,True,0,False
2023-24,218701,16,True,True,True,0,False


## 4. Compare player and team IDs separately

The identity version adds only player and team IDs, allowing a fair comparison with the context-only model.

In [5]:
example_raw = raw_by_season["2023-24"]
context_features = build_features(example_raw)
identity_features = build_features(example_raw, include_identity=True)
added_identity_columns = [column for column in identity_features if column not in context_features]

assert added_identity_columns == list(IDENTITY_CATEGORICAL_FEATURES)
assert identity_features.shape[1] == context_features.shape[1] + 2

pd.Series(
    {
        "context_only_feature_count": context_features.shape[1],
        "identity_ablation_feature_count": identity_features.shape[1],
        "columns_added_for_ablation": added_identity_columns,
    }
)

context_only_feature_count                           16
identity_ablation_feature_count                      18
columns_added_for_ablation         [player_id, team_id]
dtype: object

## 5. Feature choices

- Use 16 context features: 11 numeric and 5 categorical.
- Express coordinates in feet for readability, while preserving the source's reported shot distance.
- Add a squared distance term so the logistic baseline can represent a simple curve rather than assuming a strictly linear distance effect.
- Add angle, centerline distance, corner-three, overtime, and final-five-seconds features because the EDA showed meaningful spatial and period-clock differences.
- Do not add a coordinate-derived distance because it is almost identical to `SHOT_DISTANCE`.
- Exclude outcome fields, game/event identifiers, dates, season labels, and constant fields from predictors.
- Keep player and team IDs out of the primary feature set; add them only through `include_identity=True` for the CatBoost comparison.
- Fit all scaling and categorical encoders inside each training fold. The feature function itself learns nothing from the data.